In [51]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
text = "Some text that I'll pre-tokenize.I'll.  <|endoftext|>apple apple apple apple"
re.findall(PAT, text)

['Some',
 ' text',
 ' that',
 ' I',
 "'ll",
 ' pre',
 '-',
 'tokenize',
 '.',
 'I',
 "'ll",
 '.',
 ' ',
 ' <|',
 'endoftext',
 '|>',
 'apple',
 ' apple',
 ' apple',
 ' apple']

In [52]:
re.finditer(PAT, text)

In [53]:
max([("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")])

('BA', 'A')

In [54]:
def split_special_token(text: str, special_token: list[str]):
    if not special_token:
        yield text
        return
    sorted_specials = sorted(special_token, key=len, reverse=True)
    cursor = 0
    text_length = len(text)

    while cursor < text_length:
        next_start = None
        next_token = None
        for token in sorted_specials:
            idx = text.find(token, cursor)
            if idx != -1 and (next_start is None or idx < next_start):
                next_start = idx
                next_token = token

        if next_start is None:
            yield text[cursor:]
            return

        if next_start > cursor:
            yield text[cursor:next_start]
        cursor = next_start + len(next_token)

In [55]:
def pre_tokenize(text: str, special_token: list[str]) -> dict[tuple[bytes, ...], int]:
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    words: dict[tuple[bytes, ...], int] = {}

    for part in split_special_token(text, special_token):
        for piece in re.findall(pattern,part):
            token_tuple = tuple(bytes([byte]) for byte in piece.encode("utf-8"))
            if token_tuple:
                words[token_tuple]=words.get(token_tuple,0)+1

    return words

In [56]:
SPECIAL_TOKEN =  ["<|endoftext|>"]
text = "Some text that I'll pre-tokenize.I'll.  <|endoftext|>apple apple apple apple"
toeknized_text = pre_tokenize(text, SPECIAL_TOKEN)
print(toeknized_text)

{(b'S', b'o', b'm', b'e'): 1, (b' ', b't', b'e', b'x', b't'): 1, (b' ', b't', b'h', b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l', b'l'): 2, (b' ', b'p', b'r', b'e'): 1, (b'-',): 1, (b't', b'o', b'k', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'a', b'p', b'p', b'l', b'e'): 1, (b' ', b'a', b'p', b'p', b'l', b'e'): 3}


In [57]:
def merge_word(word: list[bytes], pair: tuple[bytes,bytes]):
    merged = []
    i=0
    while i < len(word):
        if i+1 < len(word) and (word[i], word[i + 1]) == pair:
            merged.append(word[i] + word[i + 1])
            i += 2
        else:
            merged.append(word[i])
            i += 1

    return merged

In [58]:
def merge(toeknized_text, pair):
    new_words = {}

    for word, count in toeknized_text.items():
        merged_word = tuple(merge_word(list(word), pair))
        new_words[merged_word] = new_words.get(merged_word, 0) + count

    return new_words

In [59]:
def count_pairs(words: dict[tuple[bytes, ...], int]) -> dict[tuple[bytes, bytes], int]:
    pair_counts = {}

    for word, count in words.items():
        for pair in zip(word[:-1], word[1:]):
            pair_counts[pair] = pair_counts.get(pair, 0) + count

    return pair_counts


In [60]:
pair_counts = count_pairs(toeknized_text)
print(pair_counts)

{(b'S', b'o'): 1, (b'o', b'm'): 1, (b'm', b'e'): 1, (b' ', b't'): 2, (b't', b'e'): 1, (b'e', b'x'): 1, (b'x', b't'): 1, (b't', b'h'): 1, (b'h', b'a'): 1, (b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l'): 2, (b'l', b'l'): 2, (b' ', b'p'): 1, (b'p', b'r'): 1, (b'r', b'e'): 1, (b't', b'o'): 1, (b'o', b'k'): 1, (b'k', b'e'): 1, (b'e', b'n'): 1, (b'n', b'i'): 1, (b'i', b'z'): 1, (b'z', b'e'): 1, (b' ', b' '): 1, (b'a', b'p'): 4, (b'p', b'p'): 4, (b'p', b'l'): 4, (b'l', b'e'): 4, (b' ', b'a'): 3}


In [61]:
best_pair = max(pair_counts, key=pair_counts.get)
print("best_pair =", best_pair)
print("count =", pair_counts[best_pair])


best_pair = (b'a', b'p')
count = 4


In [62]:
toeknized_text = merge(toeknized_text, best_pair)
print(toeknized_text)


{(b'S', b'o', b'm', b'e'): 1, (b' ', b't', b'e', b'x', b't'): 1, (b' ', b't', b'h', b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l', b'l'): 2, (b' ', b'p', b'r', b'e'): 1, (b'-',): 1, (b't', b'o', b'k', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'ap', b'p', b'l', b'e'): 1, (b' ', b'ap', b'p', b'l', b'e'): 3}


In [63]:
merges = []

for step in range(20):
    pair_counts = count_pairs(toeknized_text)
    if not pair_counts:
        break

    best_pair = max(pair_counts, key=pair_counts.get)
    merges.append(best_pair)

    print(f"step {step}")
    print("best_pair =", best_pair, "count =", pair_counts[best_pair])

    toeknized_text = merge(toeknized_text, best_pair)
    print(toeknized_text)
    print("-" * 40)

step 0
best_pair = (b'ap', b'p') count = 4
{(b'S', b'o', b'm', b'e'): 1, (b' ', b't', b'e', b'x', b't'): 1, (b' ', b't', b'h', b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l', b'l'): 2, (b' ', b'p', b'r', b'e'): 1, (b'-',): 1, (b't', b'o', b'k', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'app', b'l', b'e'): 1, (b' ', b'app', b'l', b'e'): 3}
----------------------------------------
step 1
best_pair = (b'app', b'l') count = 4
{(b'S', b'o', b'm', b'e'): 1, (b' ', b't', b'e', b'x', b't'): 1, (b' ', b't', b'h', b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l', b'l'): 2, (b' ', b'p', b'r', b'e'): 1, (b'-',): 1, (b't', b'o', b'k', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'appl', b'e'): 1, (b' ', b'appl', b'e'): 3}
----------------------------------------
step 2
best_pair = (b'appl', b'e') count = 4
{(b'S', b'o', b'm', b'e'): 1, (b' ', b't', b'e', b'x', b't'): 1, (b' ', b't', b'h', b'a', b't'): 1, (b' ', b'I'): 1, (b"'", b'l', b'l'

In [64]:
def pre_tokenize_all(toeknized_text):
    merges = []

    for step in range(20):
        pair_counts = count_pairs(toeknized_text)
        if not pair_counts:
            break

        best_pair = max(pair_counts, key=pair_counts.get)
        merges.append(best_pair)

        print(f"step {step}")
        print("best_pair =", best_pair, "count =", pair_counts[best_pair])

        toeknized_text = merge(toeknized_text, best_pair)
        print(toeknized_text)
        print("-" * 40)

In [65]:
pre_tokenize_all(toeknized_text)

step 0
best_pair = (b't', b'o') count = 1
{(b'Some',): 1, (b' text',): 1, (b' that',): 1, (b' I',): 1, (b"'ll",): 2, (b' pre',): 1, (b'-',): 1, (b'to', b'k', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'apple',): 1, (b' apple',): 3}
----------------------------------------
step 1
best_pair = (b'to', b'k') count = 1
{(b'Some',): 1, (b' text',): 1, (b' that',): 1, (b' I',): 1, (b"'ll",): 2, (b' pre',): 1, (b'-',): 1, (b'tok', b'e', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'apple',): 1, (b' apple',): 3}
----------------------------------------
step 2
best_pair = (b'tok', b'e') count = 1
{(b'Some',): 1, (b' text',): 1, (b' that',): 1, (b' I',): 1, (b"'ll",): 2, (b' pre',): 1, (b'-',): 1, (b'toke', b'n', b'i', b'z', b'e'): 1, (b'.',): 2, (b'I',): 1, (b' ', b' '): 1, (b'apple',): 1, (b' apple',): 3}
----------------------------------------
step 3
best_pair = (b'toke', b'n') count = 1
{(b'Some',): 1, (b' text',): 1, (b' that',): 1